Single allele groupped structue models.

In [26]:
import numpy as np

import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


 1. Read data

In [27]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/single_allele_balanced_data_base_model/" #!CHANGE for bwv!

In [28]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01',
 'HLA-B*27:05',
 'HLA-B*07:02',
 'HLA-DPA1*01:03-DPB1*04:01',
 'HLA-A*01:01',
 'HLA-DRB1*01:01',
 'HLA-A*03:01',
 'HLA-DRB1*04:01']

# 2. Prepare data and weithgs for training
## 2.1 Merge alleles into combinations

In [29]:
# combination_size = len(ALLELES)
# ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'
#                   ])
# ALLELES = sorted([ 'HLA-A*03:01'])

ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])

combination_size = 2
TOTAL_SPLITS = 5
NUM_RUNS = 5
USE_WEIGHTS = True
TARGET_LENGTHS= [13, 14, 15, 16, 17, 18]
# TARGET_LENGTHS= [15]
# TARGET_LENGTHS= [8, 9, 10, 11, 12]

In [30]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-DPA1*01:03-DPB1*04:01+HLA-DRB1*01:01
Length 9: HLA-DPA1*01:03-DPB1*04:01 - 14 HLA-DRB1*01:01 - 113 
Length 10: HLA-DPA1*01:03-DPB1*04:01 - 68 HLA-DRB1*01:01 - 123 
Length 11: HLA-DPA1*01:03-DPB1*04:01 - 288 HLA-DRB1*01:01 - 175 
Length 12: HLA-DPA1*01:03-DPB1*04:01 - 725 HLA-DRB1*01:01 - 368 
Length 13: HLA-DPA1*01:03-DPB1*04:01 - 1366 HLA-DRB1*01:01 - 826 
Length 14: HLA-DPA1*01:03-DPB1*04:01 - 1802 HLA-DRB1*01:01 - 1184 
Length 15: HLA-DPA1*01:03-DPB1*04:01 - 2260 HLA-DRB1*01:01 - 6886 
Length 16: HLA-DPA1*01:03-DPB1*04:01 - 1580 HLA-DRB1*01:01 - 1413 
Length 17: HLA-DPA1*01:03-DPB1*04:01 - 1160 HLA-DRB1*01:01 - 984 
Length 18: HLA-DPA1*01:03-DPB1*04:01 - 756 HLA-DRB1*01:01 - 708 
Length 19: HLA-DPA1*01:03-DPB1*04:01 - 460 HLA-DRB1*01:01 - 316 
Length 20: HLA-DPA1*01:03-DPB1*04:01 - 299 HLA-DRB1*01:01 - 505 
Length 21: HLA-DPA1*01:03-DPB1*04:01 - 165 HLA-DRB1*01:01 - 177 
Length 22: HLA-DPA1*01:03-DPB1*04:01 - 108 HLA-DRB1*01:01 - 80 
Length 23: HLA-DPA1*01:03-DPB1*04:01 - 74 HLA

## 2.2 Calculate counts of initial data (will be used to calculate weights further)

In [31]:
from data_reading_methods import extract_count_df, calculate_weights_based_on_length_counts

count_df_train = extract_count_df(per_allele_per_kfold_per_length_binders_train_old,
                                  selected_alleles=OLD_ALLELES,
                                  total_splits=TOTAL_SPLITS)
count_df_test = extract_count_df(per_allele_per_kfold_per_length_binders_test_old,
                                 selected_alleles=OLD_ALLELES,
                                 total_splits=TOTAL_SPLITS)


In [32]:
count_df_train.index.levels

FrozenList([['HLA-DPA1*01:03-DPB1*04:01', 'HLA-DRB1*01:01'], [0, 1, 2, 3, 4], [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30]])

## Save peptides to separate gibbs_input files just to run tests afterwards

In [33]:
import mhcnames
selected_split = 0
for old_allele_name in OLD_ALLELES:
    peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train_old[old_allele_name][selected_split][target_length] for target_length in TARGET_LENGTHS])
    target_df = pd.DataFrame({'peptide':peptides})
    target_allele_name = mhcnames.compact_allele_name(old_allele_name)
    target_df['peptide'].to_csv(f"gibbs_input/gibbs_input_{target_allele_name}.csv", index=False, header=False)

for new_allele_name in NEW_ALLELES:
    peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[new_allele_name][selected_split][target_length] for target_length in TARGET_LENGTHS])
    target_df = pd.DataFrame({'peptide':peptides})
    target_allele_name = "+".join([mhcnames.compact_allele_name(item) for item in new_allele_name.split("+")])
    target_df['peptide'].to_csv(f"gibbs_input/gibbs_input_{target_allele_name}.csv", index=False, header=False)

## 2.3 Weights for data based on counts

In [34]:
per_allele_per_kfold_per_length_weights = calculate_weights_based_on_length_counts(count_df_train,
                                                                                   selected_alleles=ALLELES,
                                                                                   total_splits=TOTAL_SPLITS,
                                                                                   selected_lengths=TARGET_LENGTHS,
                                                                                   use_weights=USE_WEIGHTS)

Total data length for split 0 is 20925
Weights for lentgh 13 are {'HLA-DPA1*01:03-DPB1*04:01': 15.318448023426061, 'HLA-DRB1*01:01': 25.332929782082324}
Weights for lentgh 14 are {'HLA-DPA1*01:03-DPB1*04:01': 11.612097669256382, 'HLA-DRB1*01:01': 17.67314189189189}
Weights for lentgh 15 are {'HLA-DPA1*01:03-DPB1*04:01': 9.258849557522124, 'HLA-DRB1*01:01': 3.0387743247168166}
Weights for lentgh 16 are {'HLA-DPA1*01:03-DPB1*04:01': 13.24367088607595, 'HLA-DRB1*01:01': 14.80891719745223}
Weights for lentgh 17 are {'HLA-DPA1*01:03-DPB1*04:01': 18.038793103448278, 'HLA-DRB1*01:01': 21.265243902439025}
Weights for lentgh 18 are {'HLA-DPA1*01:03-DPB1*04:01': 27.678571428571427, 'HLA-DRB1*01:01': 29.555084745762713}
Total data length for split 1 is 20925
Weights for lentgh 13 are {'HLA-DPA1*01:03-DPB1*04:01': 15.318448023426061, 'HLA-DRB1*01:01': 25.332929782082324}
Weights for lentgh 14 are {'HLA-DPA1*01:03-DPB1*04:01': 11.612097669256382, 'HLA-DRB1*01:01': 17.658227848101266}
Weights for le

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [35]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model_joiners_and_cycles

## 2.2 Train
### 2.2.1 Define treshold

In [11]:
stop_threshold = 0
cutoff = 0.0005
per_symbol_portion = -5
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

132.57416666666668

### 2.2.2 Prepare  initial grouped structure models

In [12]:
# Training params
algorithm = 'baum-welch'

params = dict()


params['st'] = 25
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 30
params['max_iters'] = 65
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 20
params['transition_pseudocount'] = 0
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0.25
params['distribution_inertia'] = 0
params['groups'] = 5
params['states_per_group'] = 5
params['use_pseudocount'] = True

assert params['st'] % params['groups'] == 0
assert params['st'] % params['states_per_group'] == 0
assert params['st'] == params['groups'] *  params['states_per_group']

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))



In [13]:
model = init_grouped_structure_model_joiners_and_cycles(states_per_group=int(params['states_per_group']), number_of_groups=params['groups'], freeze_distr=True)


grouped_model : grouped_model-start summed to 4.0, normalized to 1.0
grouped_model : s000 summed to 21.0, normalized to 1.0
grouped_model : s001 summed to 21.0, normalized to 1.0
grouped_model : s002 summed to 21.0, normalized to 1.0
grouped_model : s003 summed to 21.0, normalized to 1.0
grouped_model : s004 summed to 21.0, normalized to 1.0
grouped_model : s005 summed to 21.0, normalized to 1.0
grouped_model : s006 summed to 21.0, normalized to 1.0
grouped_model : s007 summed to 21.0, normalized to 1.0
grouped_model : s008 summed to 21.0, normalized to 1.0
grouped_model : s009 summed to 21.0, normalized to 1.0
grouped_model : s010 summed to 21.0, normalized to 1.0
grouped_model : s011 summed to 21.0, normalized to 1.0
grouped_model : s012 summed to 21.0, normalized to 1.0
grouped_model : s013 summed to 21.0, normalized to 1.0
grouped_model : s014 summed to 21.0, normalized to 1.0
grouped_model : s015 summed to 21.0, normalized to 1.0
grouped_model : s016 summed to 21.0, normalized to 

In [14]:
with open('model.png', 'w+') as f:
    model.plot(file=f, crop_zero=True, rotation=90, rotate=90)

In [15]:
from hmm_visualization_methods import *
plot_distributions_for_states(model,"test/", horizontal=False)

In [16]:
make_pyviz_graph(model, "test/", precision=3)

Converting graph for model (grouped_model) (Delete zero edges)


<class 'pyvis.network.Network'> |N|=31 |E|=219

In [17]:
allele_peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length] for target_length in TARGET_LENGTHS])


In [18]:
ten_mer = allele_peptides[12]
ten_mer, len(ten_mer)

('IHFHAESFLYRNG', 13)

In [15]:
cycle_state = model.states[27]
start = model.states[29]

In [17]:
model.graph.edges()[cycle_state, cycle_state]

{'probability': -0.18232155679395468, 'pseudocount': 100, 'group': None}

In [18]:
prob, path = model.viterbi(ten_mer)

NameError: name 'ten_mer' is not defined

In [62]:
[item[0] for item in path]

[29, 25, 1, 0, 7, 13, 19, 24, 26, 28, 28, 28, 28, 28, 30]

In [63]:
sixteen_mer = allele_peptides[18000]
sixteen_mer, len(sixteen_mer)

('EGELRILASTQFEPTAA', 17)

In [64]:
pd.DataFrame(model.forward_backward(sixteen_mer)[1])

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-0.076298,-inf,-2.611021,-inf
1,-2.814556,-1.507814,-2.278264,-1.527987,-1.121528,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-6.747029,-inf,-2.627136,-inf
2,-3.637920,-2.445367,-2.349157,-1.976970,-2.815598,-2.342351,-2.333283,-2.264969,-1.933892,-2.533121,...,-inf,-inf,-inf,-inf,-inf,-inf,-3.646934,-inf,-3.074468,-inf
3,-2.741252,-4.977403,-2.434469,-3.390859,-6.325422,-1.475776,-2.016867,-2.472038,-2.388709,-1.955088,...,-inf,-inf,-inf,-inf,-inf,-inf,-3.092539,-inf,-7.096927,-inf
4,-3.806397,-3.753397,-3.177733,-2.909764,-5.355614,-3.513091,-2.227211,-2.296814,-1.703493,-1.760610,...,-5.688757,-inf,-inf,-inf,-inf,-inf,-9.642228,-inf,-7.178625,-inf
5,-5.077385,-4.671229,-5.597962,-5.888652,-3.513812,-1.816470,-2.350017,-2.398214,-2.061883,-2.535318,...,-2.989226,-7.035025,-7.882222,-6.605541,-7.699761,-6.996996,-7.691761,-inf,-8.091459,-inf
6,-5.832674,-7.954722,-4.969900,-5.803428,-10.018676,-2.160929,-2.455718,-2.977390,-2.788520,-2.350743,...,-4.034453,-4.719450,-6.779471,-5.220987,-5.380037,-5.697483,-8.115255,-12.924689,-11.841601,-inf
7,-6.260026,-6.400053,-8.391015,-9.718898,-6.456727,-4.530267,-4.021441,-3.134939,-3.062403,-3.177421,...,-2.830099,-4.349325,-5.064422,-4.672997,-4.058435,-3.841425,-12.848146,-14.297908,-12.296486,-12.924689
8,-7.531404,-10.616032,-8.430893,-6.940895,-8.119146,-4.206824,-5.137531,-4.209100,-5.056503,-10.741805,...,-3.742340,-3.668550,-4.214018,-3.028794,-2.950440,-3.707778,-12.498775,-11.249130,-13.993984,-12.698917
9,-9.122135,-10.007088,-9.251858,-12.911082,-8.985154,-6.445859,-6.774037,-5.916337,-6.800559,-4.857181,...,-2.408098,-3.541158,-2.797025,-5.301754,-4.592632,-2.078851,-14.249903,-7.705496,-15.482109,-11.038366


In [65]:
pd.DataFrame(model.forward_backward(ten_mer)[1])

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-0.362429,-inf,-1.190673,-inf
1,-2.286991,-0.970776,-2.082617,-2.510640,-4.625074,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-1.572310,-inf,-2.338716,-inf
2,-1.542819,-2.420552,-1.838608,-3.499990,-1.957207,-2.426353,-3.501054,-6.454644,-2.371455,-2.865275,...,-inf,-inf,-inf,-inf,-inf,-inf,-2.793444,-inf,-3.345537,-inf
3,-3.229979,-1.733425,-2.788392,-3.139015,-5.423895,-4.234847,-1.753739,-1.434235,-4.016694,-3.059442,...,-inf,-inf,-inf,-inf,-inf,-inf,-3.531034,-inf,-5.121572,-inf
4,-2.498834,-3.582038,-4.427018,-5.997501,-2.946979,-3.460011,-3.270490,-2.505325,-2.205022,-2.239981,...,-4.286434,-inf,-inf,-inf,-inf,-inf,-7.215221,-inf,-5.253090,-inf
5,-7.158280,-5.318557,-5.069591,-4.769504,-5.797493,-2.582548,-2.718493,-2.856562,-2.517823,-3.006249,...,-2.772217,-5.416204,-4.614502,-5.048267,-5.014296,-4.849353,-5.726763,-inf,-6.227833,-inf
6,-5.411088,-8.779774,-6.918912,-5.551152,-6.142036,-3.954775,-4.786706,-3.551511,-4.597295,-10.064057,...,-3.930771,-3.797890,-4.677993,-3.338572,-3.344465,-3.393497,-6.227833,-9.543353,-inf,-inf
7,-6.375523,-6.311364,-6.323407,-8.289468,-6.747101,-5.187056,-5.906033,-9.017327,-4.957078,-5.328113,...,-2.259951,-5.415357,-2.456713,-4.335965,-3.037826,-2.785489,-inf,-4.187245,-inf,-9.543353
8,-inf,-inf,-inf,-inf,-inf,-5.607121,-5.977563,-6.386295,-6.302281,-5.887550,...,-2.857423,-1.864209,-4.381157,-2.721836,-2.926640,-2.921926,-inf,-2.875547,-inf,-4.182537
9,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-3.693247,-3.419335,-2.372858,-1.748998,-1.608876,-2.604425,-inf,-3.304970,-inf,-2.636032


In [66]:
state_sums = [sum(state.distribution.parameters[0].values()) for state in model.states[:-2]]
assert np.isclose(sum(state_sums), len(state_sums))

In [19]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            # prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])
            prepared_model = init_grouped_structure_model_joiners_and_cycles(states_per_group=int(params['states_per_group']), number_of_groups=params['groups'], freeze_distr=False)

            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

grouped_model : grouped_model-start summed to 4.0, normalized to 1.0
grouped_model : s000 summed to 21.0, normalized to 1.0
grouped_model : s001 summed to 21.0, normalized to 1.0
grouped_model : s002 summed to 21.0, normalized to 1.0
grouped_model : s003 summed to 21.0, normalized to 1.0
grouped_model : s004 summed to 21.0, normalized to 1.0
grouped_model : s005 summed to 21.0, normalized to 1.0
grouped_model : s006 summed to 21.0, normalized to 1.0
grouped_model : s007 summed to 21.0, normalized to 1.0
grouped_model : s008 summed to 21.0, normalized to 1.0
grouped_model : s009 summed to 21.0, normalized to 1.0
grouped_model : s010 summed to 21.0, normalized to 1.0
grouped_model : s011 summed to 21.0, normalized to 1.0
grouped_model : s012 summed to 21.0, normalized to 1.0
grouped_model : s013 summed to 21.0, normalized to 1.0
grouped_model : s014 summed to 21.0, normalized to 1.0
grouped_model : s015 summed to 21.0, normalized to 1.0
grouped_model : s016 summed to 21.0, normalized to 

In [20]:
ALLELES

['HLA-DPA1*01:03-DPB1*04:01+HLA-DRB1*01:01']

In [21]:
%%time

per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]


        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        max_iterations=params['max_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'],
                                        use_pseudocount=params['use_pseudocount'])
        
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)
        break


run 0
split 0
train size 20925
Shuffling for fairness 
Simple training: Train Length 20925, init_batch_size None
Use pseudocount is True
[1] Improvement: 1422968.0248441137	Time (s): 9.194
[2] Improvement: 57987.86186668463	Time (s): 9.172
[3] Improvement: 13625.46592978388	Time (s): 10.26
[4] Improvement: 8332.296847214922	Time (s): 14.98
[5] Improvement: 6698.277574088424	Time (s): 15.74
[6] Improvement: 5775.1244760435075	Time (s): 17.05
[7] Improvement: 5130.641847932711	Time (s): 14.81
[8] Improvement: 4647.480817951262	Time (s): 11.88
[9] Improvement: 4283.500247918069	Time (s): 11.79
[10] Improvement: 4021.6008406654	Time (s): 12.37
[11] Improvement: 3853.4945667274296	Time (s): 12.16
[12] Improvement: 3774.0909485202283	Time (s): 14.2
[13] Improvement: 3779.2627172823995	Time (s): 11.88
[14] Improvement: 3863.8554884418845	Time (s): 12.04
[15] Improvement: 4018.564010683447	Time (s): 13.41
[16] Improvement: 4225.527804810554	Time (s): 12.14
[17] Improvement: 4454.581040151417	T

In [22]:
from hmm_visualization_methods import *

In [23]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
Wall time: 2.01 s


In [24]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
Wall time: 13.8 s


In [25]:
from hmm_visualization_methods import *
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0) (Delete zero edges)


In [72]:
TARGET_PATH_TO_RESULTS

'experiment_results/single_allele_balanced_data_base_model/'